<a href="https://colab.research.google.com/github/auliatauhid/Data-Science-2026/blob/main/Pertemuan10_AuliaTauhid_250401020136.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd

# Membaca dataset Telco Customer Churn langsung dari URL publik (IBM sample dataset)
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Ukuran data
print(df.shape)

# Tipe data tiap kolom
print(df.dtypes)

# Cek missing value
print(df.isnull().sum())

# TotalCharges sering terbaca sebagai object karena ada spasi kosong, perlu dikonversi
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

# Proporsi kelas target (cek imbalance)
print(df["Churn"].value_counts(normalize=True))

(7043, 21)
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod

/tmp/ipykernel_2171/611753381.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)


In [10]:
from sklearn.model_selection import train_test_split

# Hapus kolom ID (tidak informatif untuk prediksi)
df = df.drop(columns=["customerID"], errors="ignore")

# Ubah target menjadi biner (Yes=1, No=0)
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Pisahkan fitur (X) dan target (y)
y = df["Churn"]
X = df.drop(columns=["Churn"])

# Encoding fitur kategorikal dengan one-hot encoding
X = pd.get_dummies(X, drop_first=True)

# Split data secara stratified (proporsi churn tetap sama di train & test)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(X_tr.shape, X_te.shape)
print(y_tr.value_counts(normalize=True))
print(y_te.value_counts(normalize=True))

(5634, 30) (1409, 30)
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


In [11]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42)

rf.fit(X_tr, y_tr)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [12]:
from sklearn.metrics import classification_report, roc_auc_score

# Prediksi kelas dan probabilitas
y_pred = rf.predict(X_te)
y_proba = rf.predict_proba(X_te)[:, 1]  # probabilitas kelas churn (1)

# Classification report (fokus ke kelas 1 = churn)
print(classification_report(y_te, y_pred, target_names=["No Churn", "Churn"]))

# ROC-AUC score
auc = roc_auc_score(y_te, y_proba)
print(f"ROC-AUC: {auc:.4f}")

              precision    recall  f1-score   support

    No Churn       0.83      0.89      0.86      1035
       Churn       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC: 0.8246


In [13]:
# Hitung probabilitas churn untuk seluruh data uji
churn_proba = rf.predict_proba(X_te)[:, 1]

# Gabungkan ke dataframe untuk dilihat
hasil = X_te.copy()
hasil["Actual_Churn"] = y_te.values
hasil["Prob_Churn"] = churn_proba

# Urutkan pelanggan dari risiko churn tertinggi
top_risk = hasil.sort_values("Prob_Churn", ascending=False).head(10)
print(top_risk[["Actual_Churn", "Prob_Churn"]])

# Feature importance (opsional, untuk insight tambahan)
importances = pd.Series(rf.feature_importances_, index=X.columns)
print(importances.sort_values(ascending=False).head(10))

      Actual_Churn  Prob_Churn
1731             1    1.000000
2194             1    0.993333
809              1    0.990000
6623             1    0.990000
1739             1    0.986667
2927             0    0.986667
3346             0    0.970000
1144             1    0.963333
4585             1    0.960000
2729             1    0.950000
TotalCharges                      0.177844
tenure                            0.164403
MonthlyCharges                    0.151054
Contract_Two year                 0.059944
InternetService_Fiber optic       0.042323
PaymentMethod_Electronic check    0.036455
Contract_One year                 0.029412
OnlineSecurity_Yes                0.028447
gender_Male                       0.025604
PaperlessBilling_Yes              0.024087
dtype: float64


In [14]:
from sklearn.metrics import classification_report, roc_auc_score

# --- Ambil ulang metrik evaluasi dalam bentuk dictionary agar bisa diolah otomatis ---
report_dict = classification_report(y_te, y_pred, target_names=["No Churn", "Churn"], output_dict=True)

precision_churn = report_dict["Churn"]["precision"]
recall_churn    = report_dict["Churn"]["recall"]
f1_churn        = report_dict["Churn"]["f1-score"]
accuracy        = report_dict["accuracy"]
auc             = roc_auc_score(y_te, y_proba)

# --- Ambil 3 fitur paling penting secara otomatis dari model ---
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
top3_features = importances.head(3)
top_feature_names = ", ".join(top3_features.index.tolist())

# --- Kategorisasi kualitatif otomatis berdasarkan nilai (bukan hardcode angka) ---
def kategori(nilai, batas_baik=0.75, batas_sedang=0.55):
    if nilai >= batas_baik:
        return "baik"
    elif nilai >= batas_sedang:
        return "cukup"
    else:
        return "rendah"

kategori_recall = kategori(recall_churn)
kategori_precision = kategori(precision_churn)
kategori_auc = kategori(auc, batas_baik=0.80, batas_sedang=0.65)

# --- Susun kesimpulan otomatis ---
kesimpulan = f"""
KESIMPULAN OTOMATIS BERDASARKAN HASIL MODEL:

1. Model Random Forest yang dibangun mencapai recall sebesar {recall_churn:.2%} pada kelas churn,
   yang tergolong {kategori_recall}, artinya model {"mampu menangkap sebagian besar" if kategori_recall != "rendah" else "masih kesulitan menangkap"}
   pelanggan yang benar-benar akan churn.

2. Precision pada kelas churn sebesar {precision_churn:.2%} (tergolong {kategori_precision}) dan F1-score
   sebesar {f1_churn:.2%}, menunjukkan {"keseimbangan yang cukup baik" if kategori_precision != "rendah" else "masih banyak false positive"}
   antara prediksi benar dan salah pada pelanggan berisiko churn.

3. Nilai ROC-AUC sebesar {auc:.4f} tergolong {kategori_auc}, yang berarti model
   {"mampu" if kategori_auc != "rendah" else "kurang mampu"} membedakan pelanggan yang akan churn
   dan yang tidak secara keseluruhan (accuracy keseluruhan model: {accuracy:.2%}).

4. Tiga fitur paling berpengaruh terhadap prediksi churn pada dataset ini adalah:
   {top_feature_names}. Tim retensi disarankan memprioritaskan pelanggan dengan
   probabilitas churn tinggi terkait pola pada fitur-fitur tersebut untuk tindakan
   pencegahan lebih awal.
"""

print(kesimpulan)


KESIMPULAN OTOMATIS BERDASARKAN HASIL MODEL:

1. Model Random Forest yang dibangun mencapai recall sebesar 50.00% pada kelas churn,
   yang tergolong rendah, artinya model masih kesulitan menangkap
   pelanggan yang benar-benar akan churn.

2. Precision pada kelas churn sebesar 62.96% (tergolong cukup) dan F1-score
   sebesar 55.74%, menunjukkan keseimbangan yang cukup baik
   antara prediksi benar dan salah pada pelanggan berisiko churn.

3. Nilai ROC-AUC sebesar 0.8246 tergolong baik, yang berarti model
   mampu membedakan pelanggan yang akan churn
   dan yang tidak secara keseluruhan (accuracy keseluruhan model: 78.92%).

4. Tiga fitur paling berpengaruh terhadap prediksi churn pada dataset ini adalah:
   TotalCharges, tenure, MonthlyCharges. Tim retensi disarankan memprioritaskan pelanggan dengan
   probabilitas churn tinggi terkait pola pada fitur-fitur tersebut untuk tindakan
   pencegahan lebih awal.

